In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.2-3B"  # Change to instruct version if needed
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)

2025-03-31 19:43:10.629780: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-31 19:43:10.649591: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743450190.669206   16017 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743450190.674438   16017 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743450190.688047   16017 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
from datasets import load_dataset

# Load the MMLU dataset
dataset = load_dataset("cais/mmlu", 'all', split='test')

In [3]:
import evaluate

def generate_prompt(question, choices):
    choices_str = "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)])
    return f"{question}\n{choices_str}\nAnswer:"

In [9]:
from tqdm import tqdm

def evaluate_mmlu(model, tokenizer, dataset):
    model.eval()
    metric = evaluate.load("accuracy")
    predictions = []
    references = []
    ta

    for example in tqdm(dataset):
        question = example['question']
        choices = example['choices']
        answer = chr(65 + example['answer'])  # Convert index to A, B, C, or D

        prompt = generate_prompt(question, choices)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=1, pad_token_id=tokenizer.eos_token_id)
            pred = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().upper()

        predictions.append(pred)
        references.append(answer)

    # Calculate accuracy using evaluate
    result = metric.compute(predictions=predictions, references=references)
    print(f"Accuracy: {result['accuracy']:.4f}")

evaluate_mmlu(model, tokenizer, dataset)

100%|██████████| 100/100 [00:03<00:00, 32.80it/s]


ValueError: invalid literal for int() with base 10: 'FIND THE DEGREE FOR THE GIVEN FIELD EXTENSION Q(SQRT(2), SQRT(3), SQRT(18)) OVER Q.\nA. 0\nB. 4\nC. 2\nD. 6\nANSWER: D'

In [8]:
predictions

NameError: name 'predictions' is not defined